In [4]:
import numpy as np
import pandas as pd


# Load dataset
data = pd.read_csv("online_shoppers_intention.csv")

data["Revenue"] = data["Revenue"].astype(int)
data["Weekend"] = data["Weekend"].astype(int)

data = pd.get_dummies(data, columns=["Month", "VisitorType"], drop_first=True)
data = data.astype(float)

X = data.drop("Revenue", axis=1).to_numpy(dtype=np.float64)
y = data["Revenue"].to_numpy(dtype=np.float64).reshape(-1, 1)

# normalize
X = X / np.maximum(X.max(axis=0), 1)

# split
split = int(0.8 * len(X))
X_train = X[:split]
y_train = y[:split]
X_test = X[split:]
y_test = y[split:]

# Functions
def sigmoid(x):
    x = np.array(x, dtype=np.float64)
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(a):
    return a * (1 - a)

def relu(x):
    x = np.array(x, dtype=np.float64)
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(np.float64)

# Network structure
input_size = X_train.shape[1]
hidden1 = 32
hidden2 = 16
output = 1

np.random.seed(1)

W1 = np.random.randn(input_size, hidden1) * 0.1
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2) * 0.1
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output) * 0.1
b3 = np.zeros((1, output))

lr = 0.01
epochs = 2000

# Training
for epoch in range(epochs):

    # Forward
    z1 = X_train @ W1 + b1
    a1 = relu(z1)

    z2 = a1 @ W2 + b2
    a2 = relu(z2)

    z3 = a2 @ W3 + b3
    y_hat = sigmoid(z3)

    # Error
    error = y_hat - y_train

    # Backprop
    dz3 = error * sigmoid_derivative(y_hat)
    dW3 = a2.T @ dz3 / len(X_train)
    db3 = np.sum(dz3, axis=0, keepdims=True) / len(X_train)

    dz2 = (dz3 @ W3.T) * relu_derivative(z2)
    dW2 = a1.T @ dz2 / len(X_train)
    db2 = np.sum(dz2, axis=0, keepdims=True) / len(X_train)

    dz1 = (dz2 @ W2.T) * relu_derivative(z1)
    dW1 = X_train.T @ dz1 / len(X_train)
    db1 = np.sum(dz1, axis=0, keepdims=True) / len(X_train)

    # Update
    W3 -= lr * dW3
    b3 -= lr * db3
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if epoch % 200 == 0:
        loss = np.mean((y_hat - y_train) ** 2)
        print("Epoch:", epoch, "Loss:", loss)

# Testing
z1 = X_test @ W1 + b1
a1 = relu(z1)

z2 = a1 @ W2 + b2
a2 = relu(z2)

z3 = a2 @ W3 + b3
pred = sigmoid(z3)

pred_class = (pred > 0.5).astype(int)
accuracy = np.mean(pred_class == y_test)

print("\nTest Accuracy:", accuracy)

Epoch: 0 Loss: 0.24652028955887176
Epoch: 200 Loss: 0.21474572160912786
Epoch: 400 Loss: 0.19083539061207158
Epoch: 600 Loss: 0.17305003902553015
Epoch: 800 Loss: 0.15988394056392322
Epoch: 1000 Loss: 0.1501694842926347
Epoch: 1200 Loss: 0.1430203289358547
Epoch: 1400 Loss: 0.13776751175993668
Epoch: 1600 Loss: 0.1339092103266295
Epoch: 1800 Loss: 0.1310712557220673

Test Accuracy: 0.7996755879967559
